# Performance Linking Prototype

This notebook prototypes the algorithm for linking extracted materials (Pipeline A)
to their performance plot data (Pipeline B) via LLM-based series matching.

**Phase 1:** Hardcoded test data + mock matcher (no API calls)
**Phase 2:** Real Gemini LLM call via DSPy
**Phase 3:** Real pipeline outputs from a test paper

## Cell 1: Imports + Pydantic Models

In [1]:
from __future__ import annotations
import json
from pydantic import BaseModel, Field


# --- Data Models ---

class SeriesMapping(BaseModel):
    """A single mapping from a plot series name to a material name."""
    series_name: str = Field(description="Series/line name from the plot legend")
    material_name: str = Field(description="Matched material name from Pipeline A")
    confidence: str = Field(description="high, medium, or low", default="medium")
    reasoning: str = Field(description="Why this match was made", default="")


class PlotMaterialMapping(BaseModel):
    """All series-to-material mappings for a single plot."""
    plot_index: int = Field(description="Index of the plot in the paper's plot list")
    figure_reference: str = Field(description="e.g. 'Fig. 3a'", default="")
    mappings: list[SeriesMapping] = Field(default_factory=list)
    unmatched_series: list[str] = Field(
        default_factory=list,
        description="Series that could not be matched (baselines, references, etc.)"
    )


class MaterialPlotEntry(BaseModel):
    """One plot series linked to a material, with its coordinate data."""
    plot_index: int
    figure_reference: str = ""
    series_name: str
    coordinates: list[list[float]] = Field(default_factory=list)
    x_axis_label: str | None = None
    x_axis_unit: str | None = None
    y_axis_label: str | None = None
    y_axis_unit: str | None = None
    plot_title: str | None = None
    confidence: str = "medium"


class MaterialPerformanceData(BaseModel):
    """All performance data for a single material, aggregated across plots."""
    material_name: str
    plot_data: list[MaterialPlotEntry] = Field(default_factory=list)


print("Models defined.")

Models defined.


## Cell 2: Hardcoded Test Data (Mo2 paper example)

In [2]:
# M: Materials extracted by Pipeline A
materials = [
    "beta-Mo2N500",
    "Mo2(C,N)Tx",
    "Mo2(C,N)T",
    "Mo2(C,N)Tx-575",
    "Mo2(C,N)Tx-Vc",
]

# P: Plots extracted by Pipeline B
plot_1 = {
    "name_to_coordinates": {
        "575": [[0.5, 210], [1, 195], [2, 170], [5, 140], [10, 105]],
        "Vc": [[0.5, 180], [1, 160], [2, 135], [5, 110], [10, 80]],
        "Mo2N": [[0.5, 150], [1, 130], [2, 100], [5, 75], [10, 50]],
        "Mo2(C,N)Tx": [[0.5, 190], [1, 175], [2, 155], [5, 125], [10, 95]],
        "bare CC": [[0.5, 20], [1, 18], [2, 15], [5, 10], [10, 8]],
    },
    "title": "Rate Performance",
    "x_axis_label": "Current density",
    "x_axis_unit": "A/g",
    "y_left_axis_label": "Specific capacitance",
    "y_left_axis_unit": "F/g",
}

plot_2 = {
    "name_to_coordinates": {
        "Mo2(C,N)Tx-575": [[1, 100], [500, 97.5], [1000, 95.2], [3000, 93.1], [5000, 91.8]],
        "Mo2(C,N)T": [[1, 100], [500, 94.0], [1000, 90.5], [3000, 85.2], [5000, 80.1]],
        "pristine Mo2N": [[1, 100], [500, 88.0], [1000, 82.0], [3000, 72.5], [5000, 65.0]],
    },
    "title": "Cycling Stability",
    "x_axis_label": "Cycle number",
    "x_axis_unit": "",
    "y_left_axis_label": "Capacitance retention",
    "y_left_axis_unit": "%",
}

plots = [plot_1, plot_2]

# C: Figure context (caption + surrounding text)
contexts = [
    "Fig 3a. Rate performance of Mo2(C,N)Tx samples and beta-Mo2N500 "
    "at various current densities on carbon cloth substrate. "
    "The Mo2(C,N)Tx-575 sample shows the highest capacitance across all rates.",
    
    "Fig 4b. Long-term cycling stability at 5 A/g for 5000 cycles. "
    "Mo2(C,N)Tx-575 retains 91.8% of initial capacitance after 5000 cycles, "
    "outperforming Mo2(C,N)T and the pristine beta-Mo2N500 precursor.",
]

print(f"Materials: {len(materials)}")
print(f"Plots: {len(plots)}")
for i, p in enumerate(plots):
    print(f"  Plot {i}: '{p['title']}' with series {list(p['name_to_coordinates'].keys())}")

Materials: 5
Plots: 2
  Plot 0: 'Rate Performance' with series ['575', 'Vc', 'Mo2N', 'Mo2(C,N)Tx', 'bare CC']
  Plot 1: 'Cycling Stability' with series ['Mo2(C,N)Tx-575', 'Mo2(C,N)T', 'pristine Mo2N']


## Cell 3: Prompt Builder + Mock Matcher

In [3]:
def build_matching_prompt(
    materials: list[str],
    series_names: list[str],
    context: str,
    plot: dict,
) -> str:
    """Build the LLM prompt for series-to-material matching."""
    return f"""You are given materials studied in a scientific paper and series/line names
extracted from a plot in that paper. Match each series name to the material
it represents.

Materials: {json.dumps(materials)}
Series names from plot: {json.dumps(series_names)}
Figure context: {context}
Plot title: {plot.get('title', 'N/A')}
X-axis: {plot.get('x_axis_label', 'N/A')} ({plot.get('x_axis_unit', '')})
Y-axis: {plot.get('y_left_axis_label', 'N/A')} ({plot.get('y_left_axis_unit', '')})

Return a JSON list of matches. Each match must have:
- "series_name": exactly one of the series names listed above
- "material_name": exactly one of the material names listed above
- "confidence": "high", "medium", or "low"
- "reasoning": brief explanation of why this match was made

Rules:
- Only match if you are confident. Skip uncertain matches.
- If a series is a baseline, reference, or substrate (not a synthesized material) — do NOT include it.
- series_name and material_name MUST be exactly from the lists above. Do not modify them.

Return ONLY a valid JSON list, no other text."""


def mock_match(series_names: list[str], materials: list[str]) -> list[dict]:
    """Mock matcher for Phase 1 testing. Simulates expected LLM output."""
    # Hardcoded mock responses for our test data
    known_mappings = {
        "575": ("Mo2(C,N)Tx-575", "high", "575 is the annealing temperature suffix"),
        "Vc": ("Mo2(C,N)Tx-Vc", "high", "Vc matches the vacuum-annealed variant"),
        "Mo2N": ("beta-Mo2N500", "medium", "Mo2N is shorthand for the beta-Mo2N phase"),
        "Mo2(C,N)Tx": ("Mo2(C,N)Tx", "high", "Direct name match"),
        "Mo2(C,N)Tx-575": ("Mo2(C,N)Tx-575", "high", "Direct name match"),
        "Mo2(C,N)T": ("Mo2(C,N)T", "high", "Direct name match"),
        "pristine Mo2N": ("beta-Mo2N500", "medium", "Pristine Mo2N refers to the precursor beta-Mo2N500"),
        # "bare CC" is intentionally NOT mapped (it's a substrate)
    }
    
    result = []
    for sn in series_names:
        if sn in known_mappings:
            mat, conf, reason = known_mappings[sn]
            if mat in materials:  # only include if material is valid
                result.append({
                    "series_name": sn,
                    "material_name": mat,
                    "confidence": conf,
                    "reasoning": reason,
                })
    return result


# Test the prompt
prompt = build_matching_prompt(
    materials,
    list(plot_1["name_to_coordinates"].keys()),
    contexts[0],
    plot_1,
)
print(prompt)
print("\n" + "="*60)
print("\nMock response for plot_1:")
mock_result = mock_match(list(plot_1["name_to_coordinates"].keys()), materials)
print(json.dumps(mock_result, indent=2))

You are given materials studied in a scientific paper and series/line names
extracted from a plot in that paper. Match each series name to the material
it represents.

Materials: ["beta-Mo2N500", "Mo2(C,N)Tx", "Mo2(C,N)T", "Mo2(C,N)Tx-575", "Mo2(C,N)Tx-Vc"]
Series names from plot: ["575", "Vc", "Mo2N", "Mo2(C,N)Tx", "bare CC"]
Figure context: Fig 3a. Rate performance of Mo2(C,N)Tx samples and beta-Mo2N500 at various current densities on carbon cloth substrate. The Mo2(C,N)Tx-575 sample shows the highest capacitance across all rates.
Plot title: Rate Performance
X-axis: Current density (A/g)
Y-axis: Specific capacitance (F/g)

Return a JSON list of matches. Each match must have:
- "series_name": exactly one of the series names listed above
- "material_name": exactly one of the material names listed above
- "confidence": "high", "medium", or "low"
- "reasoning": brief explanation of why this match was made

Rules:
- Only match if you are confident. Skip uncertain matches.
- If a series i

## Cell 4: Validation Logic

In [4]:
def validate_mappings(
    raw_mappings: list[dict],
    valid_series: list[str],
    valid_materials: list[str],
) -> tuple[list[SeriesMapping], list[str]]:
    """
    Validate LLM-returned mappings:
    - Discard any mapping with hallucinated series_name or material_name
    - Return validated mappings + list of unmatched series
    """
    valid_series_set = set(valid_series)
    valid_materials_set = set(valid_materials)
    
    validated = []
    matched_series = set()
    
    for m in raw_mappings:
        sn = m.get("series_name", "")
        mn = m.get("material_name", "")
        
        if sn not in valid_series_set:
            print(f"  [WARN] Discarding mapping: series '{sn}' not in plot series list")
            continue
        if mn not in valid_materials_set:
            print(f"  [WARN] Discarding mapping: material '{mn}' not in materials list")
            continue
        
        validated.append(SeriesMapping(
            series_name=sn,
            material_name=mn,
            confidence=m.get("confidence", "medium"),
            reasoning=m.get("reasoning", ""),
        ))
        matched_series.add(sn)
    
    unmatched = [s for s in valid_series if s not in matched_series]
    
    return validated, unmatched


# Test validation
test_raw = mock_match(list(plot_1["name_to_coordinates"].keys()), materials)
# Add a hallucinated entry to test filtering
test_raw.append({"series_name": "FAKE_SERIES", "material_name": "FAKE_MAT", "confidence": "high", "reasoning": "hallucinated"})

validated, unmatched = validate_mappings(
    test_raw,
    list(plot_1["name_to_coordinates"].keys()),
    materials,
)
print(f"\nValidated: {len(validated)} mappings")
for v in validated:
    print(f"  {v.series_name} -> {v.material_name} ({v.confidence})")
print(f"Unmatched: {unmatched}")

  [WARN] Discarding mapping: series 'FAKE_SERIES' not in plot series list

Validated: 4 mappings
  575 -> Mo2(C,N)Tx-575 (high)
  Vc -> Mo2(C,N)Tx-Vc (high)
  Mo2N -> beta-Mo2N500 (medium)
  Mo2(C,N)Tx -> Mo2(C,N)Tx (high)
Unmatched: ['bare CC']


## Cell 5: Run Matching on All Plots

In [5]:
def match_series_to_materials(
    materials: list[str],
    series_names: list[str],
    context: str,
    plot: dict,
    llm=None,
) -> list[dict]:
    """
    Match plot series names to material names.
    If llm=None, uses mock matcher (Phase 1).
    If llm is provided, calls the LLM (Phase 2).
    """
    if llm is None:
        return mock_match(series_names, materials)
    
    # Phase 2: real LLM call
    prompt = build_matching_prompt(materials, series_names, context, plot)
    response = llm(prompt)
    
    # Parse JSON from response
    # The LLM should return a JSON list, but may wrap it in markdown code blocks
    response_text = response if isinstance(response, str) else response[0]
    response_text = response_text.strip()
    if response_text.startswith("```"):
        # Strip markdown code block
        lines = response_text.split("\n")
        response_text = "\n".join(lines[1:-1])
    
    try:
        return json.loads(response_text)
    except json.JSONDecodeError as e:
        print(f"  [ERROR] Failed to parse LLM response as JSON: {e}")
        print(f"  Response was: {response_text[:500]}")
        return []


def link_all_plots(
    materials: list[str],
    plots: list[dict],
    contexts: list[str],
    llm=None,
) -> list[PlotMaterialMapping]:
    """Run matching for each plot, return list of PlotMaterialMapping."""
    all_mappings = []
    
    for idx, (plot, ctx) in enumerate(zip(plots, contexts)):
        series_names = list(plot["name_to_coordinates"].keys())
        print(f"\nPlot {idx}: '{plot.get('title', 'N/A')}' — {len(series_names)} series")
        
        raw = match_series_to_materials(materials, series_names, ctx, plot, llm)
        validated, unmatched = validate_mappings(raw, series_names, materials)
        
        all_mappings.append(PlotMaterialMapping(
            plot_index=idx,
            figure_reference=f"Plot {idx}",
            mappings=validated,
            unmatched_series=unmatched,
        ))
        
        print(f"  Matched: {len(validated)}, Unmatched: {unmatched}")
    
    return all_mappings


# Run with mock matcher
all_mappings = link_all_plots(materials, plots, contexts)


Plot 0: 'Rate Performance' — 5 series
  Matched: 4, Unmatched: ['bare CC']

Plot 1: 'Cycling Stability' — 3 series
  Matched: 3, Unmatched: []


## Cell 6: Aggregate Per Material

In [6]:
def aggregate_performance(
    material_name: str,
    mappings: list[PlotMaterialMapping],
    plots: list[dict],
) -> MaterialPerformanceData:
    """Collect all plot data for a single material across all plots."""
    entries = []
    
    for mapping in mappings:
        plot = plots[mapping.plot_index]
        for sm in mapping.mappings:
            if sm.material_name == material_name:
                coords = plot["name_to_coordinates"].get(sm.series_name, [])
                entries.append(MaterialPlotEntry(
                    plot_index=mapping.plot_index,
                    figure_reference=mapping.figure_reference,
                    series_name=sm.series_name,
                    coordinates=coords,
                    x_axis_label=plot.get("x_axis_label"),
                    x_axis_unit=plot.get("x_axis_unit"),
                    y_axis_label=plot.get("y_left_axis_label"),
                    y_axis_unit=plot.get("y_left_axis_unit"),
                    plot_title=plot.get("title"),
                    confidence=sm.confidence,
                ))
    
    return MaterialPerformanceData(
        material_name=material_name,
        plot_data=entries,
    )


# Build per-material performance for all materials
print("=" * 70)
print("PER-MATERIAL PERFORMANCE SUMMARY")
print("=" * 70)

all_performance = {}
for mat in materials:
    perf = aggregate_performance(mat, all_mappings, plots)
    all_performance[mat] = perf
    
    print(f"\n{mat}: {len(perf.plot_data)} performance entries")
    if not perf.plot_data:
        print("  (no performance data linked)")
    for entry in perf.plot_data:
        print(
            f"  - {entry.plot_title} / series '{entry.series_name}' "
            f"({entry.y_axis_label} [{entry.y_axis_unit}] vs "
            f"{entry.x_axis_label} [{entry.x_axis_unit}]), "
            f"{len(entry.coordinates)} points, confidence: {entry.confidence}"
        )

PER-MATERIAL PERFORMANCE SUMMARY

beta-Mo2N500: 2 performance entries
  - Rate Performance / series 'Mo2N' (Specific capacitance [F/g] vs Current density [A/g]), 5 points, confidence: medium
  - Cycling Stability / series 'pristine Mo2N' (Capacitance retention [%] vs Cycle number []), 5 points, confidence: medium

Mo2(C,N)Tx: 1 performance entries
  - Rate Performance / series 'Mo2(C,N)Tx' (Specific capacitance [F/g] vs Current density [A/g]), 5 points, confidence: high

Mo2(C,N)T: 1 performance entries
  - Cycling Stability / series 'Mo2(C,N)T' (Capacitance retention [%] vs Cycle number []), 5 points, confidence: high

Mo2(C,N)Tx-575: 2 performance entries
  - Rate Performance / series '575' (Specific capacitance [F/g] vs Current density [A/g]), 5 points, confidence: high
  - Cycling Stability / series 'Mo2(C,N)Tx-575' (Capacitance retention [%] vs Cycle number []), 5 points, confidence: high

Mo2(C,N)Tx-Vc: 1 performance entries
  - Rate Performance / series 'Vc' (Specific capacitanc

## Cell 7: Build Combined Synthesis + Performance Objects

In [7]:
# Simulate synthesis data from Pipeline A (abbreviated for prototype)
mock_syntheses = {
    "beta-Mo2N500": {
        "target_compound": "beta-Mo2N500",
        "synthesis_method": "solid-state",
        "target_compound_type": "two-dimensional materials",
    },
    "Mo2(C,N)Tx": {
        "target_compound": "Mo2(C,N)Tx",
        "synthesis_method": "other",
        "target_compound_type": "two-dimensional materials",
    },
    "Mo2(C,N)T": {
        "target_compound": "Mo2(C,N)T",
        "synthesis_method": "other",
        "target_compound_type": "two-dimensional materials",
    },
    "Mo2(C,N)Tx-575": {
        "target_compound": "Mo2(C,N)Tx-575",
        "synthesis_method": "calcine",
        "target_compound_type": "two-dimensional materials",
    },
    "Mo2(C,N)Tx-Vc": {
        "target_compound": "Mo2(C,N)Tx-Vc",
        "synthesis_method": "calcine",
        "target_compound_type": "two-dimensional materials",
    },
}


# Build combined objects
combined_results = []
for mat in materials:
    perf = all_performance[mat]
    synth = mock_syntheses.get(mat, {})
    
    combined = {
        "material": mat,
        "synthesis": synth,
        "performance": perf.model_dump() if perf.plot_data else None,
    }
    combined_results.append(combined)

# Pretty print one example
print("=" * 70)
print("EXAMPLE: Combined object for Mo2(C,N)Tx-575")
print("=" * 70)
example = next(r for r in combined_results if r["material"] == "Mo2(C,N)Tx-575")
print(json.dumps(example, indent=2))

EXAMPLE: Combined object for Mo2(C,N)Tx-575
{
  "material": "Mo2(C,N)Tx-575",
  "synthesis": {
    "target_compound": "Mo2(C,N)Tx-575",
    "synthesis_method": "calcine",
    "target_compound_type": "two-dimensional materials"
  },
  "performance": {
    "material_name": "Mo2(C,N)Tx-575",
    "plot_data": [
      {
        "plot_index": 0,
        "figure_reference": "Plot 0",
        "series_name": "575",
        "coordinates": [
          [
            0.5,
            210.0
          ],
          [
            1.0,
            195.0
          ],
          [
            2.0,
            170.0
          ],
          [
            5.0,
            140.0
          ],
          [
            10.0,
            105.0
          ]
        ],
        "x_axis_label": "Current density",
        "x_axis_unit": "A/g",
        "y_axis_label": "Specific capacitance",
        "y_axis_unit": "F/g",
        "plot_title": "Rate Performance",
        "confidence": "high"
      },
      {
        "plot_i

## Cell 8: Summary Statistics

In [8]:
print("=" * 70)
print("LINKING SUMMARY")
print("=" * 70)

total_series = sum(len(p["name_to_coordinates"]) for p in plots)
total_matched = sum(len(m.mappings) for m in all_mappings)
total_unmatched = sum(len(m.unmatched_series) for m in all_mappings)

print(f"Materials: {len(materials)}")
print(f"Plots: {len(plots)}")
print(f"Total series across all plots: {total_series}")
print(f"Matched series: {total_matched}")
print(f"Unmatched series: {total_unmatched}")
print(f"Match rate: {total_matched/total_series*100:.1f}%")

print(f"\nPer-material coverage:")
for mat in materials:
    n = len(all_performance[mat].plot_data)
    print(f"  {mat}: {n} performance entries")

print(f"\nUnmatched series (baselines/references):")
for m in all_mappings:
    if m.unmatched_series:
        print(f"  Plot {m.plot_index}: {m.unmatched_series}")

LINKING SUMMARY
Materials: 5
Plots: 2
Total series across all plots: 8
Matched series: 7
Unmatched series: 1
Match rate: 87.5%

Per-material coverage:
  beta-Mo2N500: 2 performance entries
  Mo2(C,N)Tx: 1 performance entries
  Mo2(C,N)T: 1 performance entries
  Mo2(C,N)Tx-575: 2 performance entries
  Mo2(C,N)Tx-Vc: 1 performance entries

Unmatched series (baselines/references):
  Plot 0: ['bare CC']


---
## Phase 2: Real Gemini LLM Call

Uncomment and run the cells below once you want to test with a real LLM.

In [ ]:
# Phase 2: Real Gemini call
import os
from dotenv import load_dotenv

load_dotenv("/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/.env")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")

import dspy

lm = dspy.LM("gemini/gemini-2.0-flash", temperature=0.0, api_key=os.environ["GEMINI_API_KEY"])

# Re-run with real LLM
print("Running with real Gemini LLM...\n")
all_mappings_real = link_all_plots(materials, plots, contexts, llm=lm)

# Aggregate and display
print("\n" + "=" * 70)
print("REAL LLM — PER-MATERIAL PERFORMANCE SUMMARY")
print("=" * 70)

all_performance_real = {}
for mat in materials:
    perf = aggregate_performance(mat, all_mappings_real, plots)
    all_performance_real[mat] = perf
    print(f"\n{mat}: {len(perf.plot_data)} performance entries")
    for entry in perf.plot_data:
        print(f"  - {entry.plot_title} / '{entry.series_name}' ({entry.confidence})")

# Summary
total_series = sum(len(p["name_to_coordinates"]) for p in plots)
total_matched = sum(len(m.mappings) for m in all_mappings_real)
print(f"\nMatch rate: {total_matched}/{total_series} = {total_matched/total_series*100:.1f}%")
print(f"Unmatched: {[s for m in all_mappings_real for s in m.unmatched_series]}")